# Regelleistung.net
Required data:
 - Activation Price (CBMP)
 - Capacity Price

In [2]:
import pandas as pd
import requests
import io
import warnings

# Unterdrückt die irrelevante Openpyxl-Warnung wegen fehlender Styles
warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

def fetch_and_parse_regelleistung(year, market_type):
    """
    Lädt aFRR Jahres-Daten direkt per URL herunter und formatiert sie robust.
    """
    url = f"https://www.regelleistung.net/apps/cpp-publisher/api/v2/tenders/files/RESULT_OVERVIEW_{market_type}_MARKET_aFRR_{year}-01-01_{year}-12-31.xlsx"
    
    print(f"Lade {market_type}-Daten für {year} herunter...")
    response = requests.get(url)
    response.raise_for_status() 

    df = pd.read_excel(io.BytesIO(response.content), engine='openpyxl')
    df.columns = df.columns.str.strip() # Leerzeichen entfernen

    # --- 1. DYNAMISCHE SPALTEN-SUCHE ---
    # Finde die Datums-Spalte (egal ob DATE, DELIVERY_DATE oder Datum)
    date_col = [c for c in df.columns if 'DATE' in c.upper() or 'DATUM' in c.upper()][0]
    
    # Finde die Produkt-Spalte (PRODUCT, PRODUKT, TIME_SLICE)
    prod_col = [c for c in df.columns if 'PRODUC' in c.upper() or 'PRODUKT' in c.upper() or 'TIME' in c.upper()][0]

    # --- 2. ZEITSTEMPEL PARSEN ---
    # Fall A: 15-Minuten Format (z.B. "NEG_001")
    if df[prod_col].astype(str).str.contains('_').any():
        df['quarter_hour_int'] = df[prod_col].str.extract(r'_(\d+)').astype(int)
        df['time_offset'] = pd.to_timedelta((df['quarter_hour_int'] - 1) * 15, unit='m')
        df['timestamp'] = pd.to_datetime(df[date_col]) + df['time_offset']
        df['Richtung'] = df[prod_col].str.split('_').str[0]
        
    # Fall B: Altes 4-Stunden Format (z.B. "00:00 - 04:00" in 2022 Capacity)
    else:
        df['start_time'] = df[prod_col].astype(str).str.split(' - ').str[0]
        df['timestamp'] = pd.to_datetime(df[date_col].astype(str) + ' ' + df['start_time'])
        # Richtung steht hier oft in einer extra Spalte (z.B. "RESERVE_DIRECTION")
        dir_col = [c for c in df.columns if 'DIRECTION' in c.upper() or 'RICHTUNG' in c.upper()][0]
        df['Richtung'] = df[dir_col]

    df['timestamp'] = df['timestamp'].dt.tz_localize('Europe/Berlin', ambiguous='NaT', nonexistent='NaT').dt.tz_convert('UTC')

    # --- 3. PREIS-SPALTE FINDEN ---
    if market_type == 'ENERGY':
        price_col = [c for c in df.columns if 'MARGINAL_ENERGY_PRICE' in c.upper()][0]
        name_prefix = 'afrr_activation_price'
    else:
        price_col = [c for c in df.columns if 'MARGINAL_CAPACITY_PRICE' in c.upper() or 'GRENZWERT' in c.upper()][0]
        name_prefix = 'afrr_capacity_price'

    df[price_col] = pd.to_numeric(df[price_col], errors='coerce').fillna(0.0)

    # --- 4. PIVOTISIEREN UND RESAMPLEN ---
    df_clean = df[['timestamp', 'Richtung', price_col]].dropna(subset=['timestamp'])
    df_pivot = df_clean.pivot_table(index='timestamp', columns='Richtung', values=price_col)
    
    # Sicherstellen, dass die Spalten POS und NEG heißen (manchmal heißen sie 'POSITIVE'/'NEGATIVE')
    df_pivot.columns = [col.replace('ATIVE', '') for col in df_pivot.columns] # NEGATIVE -> NEG
    df_pivot.rename(columns={'NEG': f'{name_prefix}_neg', 'POS': f'{name_prefix}_pos'}, inplace=True)
    
    # Auf 15-Minuten-Intervalle auffüllen (wichtig für die 4h-Blöcke der alten Capacity-Daten)
    df_pivot = df_pivot.resample('15min').ffill()
    
    return df_pivot


# ==============================================================================
# AUSFÜHRUNG 
# ==============================================================================
YEAR = 2022

df_cap = fetch_and_parse_regelleistung(YEAR, 'CAPACITY')
df_ene = fetch_and_parse_regelleistung(YEAR, 'ENERGY')

# Merge
df_final = pd.concat([df_cap, df_ene], axis=1).ffill()

print("\n--- ERFOLG! HIER IST DEINE MASTER-TABELLE ---")
print(df_final.head())

Lade CAPACITY-Daten für 2022 herunter...
Lade ENERGY-Daten für 2022 herunter...

--- ERFOLG! HIER IST DEINE MASTER-TABELLE ---
                           afrr_capacity_price_neg  afrr_capacity_price_pos  \
timestamp                                                                     
2021-12-31 22:45:00+00:00                    44.69                     0.98   
2021-12-31 23:00:00+00:00                    44.69                     0.98   
2021-12-31 23:15:00+00:00                    44.69                     0.98   
2021-12-31 23:30:00+00:00                    44.69                     0.98   
2021-12-31 23:45:00+00:00                    38.70                     1.55   

                           afrr_activation_price_neg  \
timestamp                                              
2021-12-31 22:45:00+00:00                   -9999.99   
2021-12-31 23:00:00+00:00                   -9999.99   
2021-12-31 23:15:00+00:00                   -9999.99   
2021-12-31 23:30:00+00:00              